In [1]:
!pip install torchxrayvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 67.0 MB/s eta 0:00:00


In [2]:
import os
import glob
import cv2
import numpy as np
import torch
import torchvision
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchxrayvision as xrv
import concurrent.futures
from tqdm.auto import tqdm
import copy
from pathlib import Path

In [3]:
class CXRSegmentationDataset(Dataset):
    def __init__(self, image_dir):
        self.image_paths = glob.glob(os.path.join(image_dir, "*.png"))
        self.transform = torchvision.transforms.Compose([
            xrv.datasets.XRayCenterCrop(),
        ])
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img_id = Path(img_path).stem
        
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = xrv.datasets.normalize(img, 255)
        img = img[None, ...] 
        img = self.transform(img)
        
        return torch.from_numpy(img).float(), img_id

In [4]:
def process_chunk_on_device(loader, model, device, worker_id, output_dir):

    model.eval()
    
    # tqdm hiển thị tiến độ độc lập cho từng GPU
    pos = int(str(device)[-1]) if 'cuda:' in str(device) else 0
    pbar = tqdm(loader, desc=f"Worker {worker_id} [{device}]", position=pos, leave=True)
    
    with torch.no_grad():
        for imgs, img_ids in pbar:
            imgs = imgs.to(device)
            
            # (B, 1, 1024, 1024) -> (B, 1, 512, 512)
            imgs_512 = F.interpolate(imgs, size=(512, 512), mode='bilinear', align_corners=False)
            
            # Predict
            pred_512 = model(imgs_512)
            
            # (B, 14, 512, 512) -> (B, 14, 1024, 1024)
            pred_1024 = F.interpolate(pred_512, size=(imgs.shape[-2], imgs.shape[-1]), mode='bilinear', align_corners=False)
            
            # Softmax to boolean mask (Threshold 0.5)
            pred_soft = 1 / (1 + torch.exp(-pred_1024))
            
            # CHÚ Ý: Đẩy về np.bool_ để ép dung lượng tensor nhỏ nhất mức có thể (còn nhỏ hơn uint8)
            pred_bin = (pred_soft > 0.5).cpu().numpy().astype(np.bool_) 
            
            # XẢ ĐĨA NGAY CHO TỪNG ẢNH BÊN TRONG BATCH
            for i, img_id in enumerate(img_ids):
                mask_vector = pred_bin[i] # Lấy ra (14, 1024, 1024) cho ảnh này
                save_path = os.path.join(OUTPUT_DIR, f"{img_id}.npz")
                
                # Lưu dưới định dạng nén (tiết kiệm không gian ổ cứng cực tốt) có 'mask' là khóa
                np.savez_compressed(save_path, mask=mask_vector)

In [5]:
def anomaly_map_generator(image_dir: str, output_dir: str, batch_size=16):

    dataset = CXRSegmentationDataset(image_dir)
    
    dataset_size = len(dataset)
    
    print(f"Tổng số ảnh: {dataset_size}")
    
    # Tải mô hình gốc lên GPU 0
    device_0 = torch.device('cuda:0')
    model_0 = xrv.baseline_models.chestx_det.PSPNet().to(device_0)
    
    num_gpus = torch.cuda.device_count()
    
    if num_gpus > 1:
        print(f"Phát hiện {num_gpus} GPUs. Kích hoạt Multi-Threading GPU!")
        device_1 = torch.device('cuda:1')
        model_1 = copy.deepcopy(model_0).to(device_1)
        
        # Chia dataset
        mid = dataset_size // 2
        indices = list(range(dataset_size))
        subset_0 = Subset(dataset, indices[:mid])
        subset_1 = Subset(dataset, indices[mid:])
        
        # Num-workers nên = 2 để tránh thắt cổ chai I/O khi đọc ảnh
        loader_0 = DataLoader(subset_0, batch_size=batch_size, shuffle=False, num_workers=2)
        loader_1 = DataLoader(subset_1, batch_size=batch_size, shuffle=False, num_workers=2)
        
        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
            future_0 = executor.submit(process_chunk_on_device, loader_0, model_0, device_0, 0, output_dir)
            future_1 = executor.submit(process_chunk_on_device, loader_1, model_1, device_1, 1, output_dir)
            # Ép chờ hai luồng hoàn tất
            future_0.result()
            future_1.result()
    else:
        print("Phát hiện 1 GPU. Chạy Mono-Thread!")
        loader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=4)
        process_chunk_on_device(loader, model_0, device_0, 0)

    print("Batch processing completed! Toàn bộ file .npz đã được lưu thành công.")

In [6]:
IMAGE_DIR = "/kaggle/input/datasets/thebeo182004/medanomaly-vindrcxr/medi_anomaly/train" 

OUTPUT_DIR = "/kaggle/working/medianomaly/train"

os.makedirs(OUTPUT_DIR, exist_ok=True)

anomaly_map_generator(image_dir=IMAGE_DIR, output_dir=OUTPUT_DIR, batch_size=25)

Tổng số ảnh: 4000
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/pspnet_chestxray_best_model_4.pth -O /root/.torchxrayvision/models_data/pspnet_chestxray_best_model_4.pth`
[██████████████████████████████████████████████████]
Phát hiện 2 GPUs. Kích hoạt Multi-Threading GPU!


Worker 1 [cuda:1]:   0%|          | 0/80 [00:00<?, ?it/s]

Worker 0 [cuda:0]:   0%|          | 0/80 [00:00<?, ?it/s]

Batch processing completed! Toàn bộ file .npz đã được lưu thành công.
